# 🚀 심화: Function Calling과 JSON 스키마

## 1. 도구 설명서 (JSON Schema) 작성
OpenAI 모델 등 최신 LLM이 파이썬 함수를 인식할 수 있도록 인자 타입과 설명을 구조화된 JSON(딕셔너리) 형태로 정의합니다.

In [1]:
# 날씨 API를 가정하는 도구의 스키마
weather_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
}

# 계산기 도구의 스키마
calculator_tool_schema = {
    "type": "function",
    "function": {
        "name": "calculate_math",
        "description": "Evaluate a mathematical expression",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g., '2 + 2 * 3'",
                }
            },
            "required": ["expression"],
        },
    }
}

tools_list = [weather_tool_schema, calculator_tool_schema]
print("✅ 모델에게 제공할 JSON 스키마(도구 설명서) 준비 완료.")


✅ 모델에게 제공할 JSON 스키마(도구 설명서) 준비 완료.


## 2. 상태 기계 (State Machine) 통신 모의 시뮬레이션
언어 모델에 API를 요청할 때 저 스키마들을 같이 던져주면, 모델이 텍스트 대신 `tool_calls` 객체를 뱉어내는 상황을 시뮬레이션합니다.

In [2]:
import json

# 가상의 파이썬 함수 실제 구현체
def get_current_weather(location, unit="celsius"):
    # 가상의 외부 서버 호출 결과라고 가정
    if "Seoul" in location:
        return {"temperature": 22, "unit": unit, "description": "Sunny"}
    else:
        return {"temperature": 15, "unit": unit, "description": "Cloudy"}

# 에이전트 엔진은 LLM API 응답을 기다립니다.
print("👨‍💻 사용자: '오늘 서울 날씨는 섭씨 몇 도야?'\n")

# --- LLM API 내부 동작 가정 ---
# LLM은 질문과 도구 스키마를 비교한 뒤, 대답 대신 함수를 호출하라는 JSON을 뱉어냅니다.
mock_llm_api_response = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_abc123",
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "arguments": '{"location": "Seoul", "unit": "celsius"}' # LLM이 뽑아낸 인자값!
            }
        }
    ]
}
# ----------------------------

print("🤖 LLM 응답: (텍스트를 생성하지 않고, 도구 사용을 요청함)")
tool_call = mock_llm_api_response["tool_calls"][0]
print(f"요청된 함수: {tool_call['function']['name']}")
print(f"추출된 인자: {tool_call['function']['arguments']}\n")

# 소프트웨어 엔진에서 함수 매핑 및 실행
if tool_call["function"]["name"] == "get_current_weather":
    # JSON 문자열을 파이썬 딕셔너리로 파싱
    arguments = json.loads(tool_call["function"]["arguments"])
    
    print("⚙️ [에이전트 엔진]: 파이썬 함수 실행 중...")
    result = get_current_weather(location=arguments["location"], unit=arguments.get("unit", "celsius"))
    print(f"👀 [함수 실행 결과]: {result}\n")
    
# --- LLM API 내부 동작 가정 (두 번째 호출) ---
# 엔진은 함수 실행 결과(Observation)를 다시 LLM에게 던져줍니다.
# LLM은 결과를 해석하여 최종 텍스트 답변을 생성합니다.
mock_final_llm_response = "오늘 서울의 날씨는 화창하며, 기온은 섭씨 22도입니다."
# ----------------------------------------

print("🤖 LLM 최종 출력:")
print(mock_final_llm_response)

print("\n💡 결론: Function Calling은 복잡한 프롬프트 파싱(ReAct) 없이도, LLM이 JSON 스키마를 이해하고 정확한 변수를 뽑아주어 개발자가 코드를 실행하게 만들어주는 우아하고 강력한 방법론입니다.")

👨‍💻 사용자: '오늘 서울 날씨는 섭씨 몇 도야?'

🤖 LLM 응답: (텍스트를 생성하지 않고, 도구 사용을 요청함)
요청된 함수: get_current_weather
추출된 인자: {"location": "Seoul", "unit": "celsius"}

⚙️ [에이전트 엔진]: 파이썬 함수 실행 중...
👀 [함수 실행 결과]: {'temperature': 22, 'unit': 'celsius', 'description': 'Sunny'}

🤖 LLM 최종 출력:
오늘 서울의 날씨는 화창하며, 기온은 섭씨 22도입니다.

💡 결론: Function Calling은 복잡한 프롬프트 파싱(ReAct) 없이도, LLM이 JSON 스키마를 이해하고 정확한 변수를 뽑아주어 개발자가 코드를 실행하게 만들어주는 우아하고 강력한 방법론입니다.
